# Topic 3: Pandas UDFs & Apache Arrow

> **Phase 7 → Week 12 → Topic 3**

---

## Why Regular UDFs Are Slow

```
Regular Python UDF:
  Spark (JVM) → serialize row as Python pickle → Python process → deserialize
                → execute Python function → serialize result → JVM → next row
  
  One row at a time. Serialization overhead > actual computation.
  Can be 10-100× slower than built-in Spark functions.

Pandas UDF (vectorized):
  Spark (JVM) → serialize BATCH of rows as Apache Arrow → Python
                → execute on entire pandas Series/DataFrame → serialize result batch → JVM
  
  Batch at a time. Arrow zero-copy serialization (minimal overhead).
  2-10× faster than regular UDFs. Still slower than built-in functions.

Built-in functions (fastest → use these first):
  No JVM↔Python boundary. Pure JVM/Catalyst execution.
```

---

## Pandas UDF Types

```
Type                     Input                Output          Use case
────────────────────────  ───────────────────  ─────────────  ─────────────────────────
Scalar                   pandas.Series        pandas.Series  Column transformation
Scalar Iterator          Iterator[Series]     Iterator[Ser]  Expensive model loading
Grouped Map (applyInPandas) pandas.DataFrame  DataFrame      Group-level transforms
Grouped Aggregate (pandas_udf GROUPED_AGG) Series → scalar  Custom aggregations
```

---

## Interview Cheat Sheet

**Q: What is Apache Arrow and what role does it play in PySpark?**
> Apache Arrow is a cross-language, columnar memory format for in-memory analytics. In PySpark, Arrow is used to transfer data between the JVM (Spark engine) and the Python process without expensive pickle serialization. Enable with `spark.sql.execution.arrow.pyspark.enabled=true`. Arrow transfers entire column batches at once rather than row-by-row — typically 10× less overhead than pickle.

**Q: What is a Pandas UDF vs a regular Python UDF?**
> A regular Python UDF processes one Row at a time via pickle serialization. A Pandas UDF (vectorized UDF) receives an entire batch of rows as a pandas Series, applies the function to the whole batch, and returns a pandas Series. The batch transfer uses Apache Arrow (zero-copy where possible). Result: 2-10× faster than regular UDFs and allows using the full pandas/NumPy/scikit-learn ecosystem.

**Q: When would you use `applyInPandas` instead of a scalar Pandas UDF?**
> `applyInPandas` passes an entire group (all rows for a given key) as a pandas DataFrame. Use it when the transformation requires all rows of a group simultaneously: (1) ML inference per group, (2) normalization within a group, (3) computing lag/lead within a group without Window functions, (4) any operation that needs cross-row context within a key.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import time

spark = SparkSession.builder \
    .appName("Week12 - Pandas UDFs and Arrow") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)
print("Arrow enabled:", spark.conf.get("spark.sql.execution.arrow.pyspark.enabled"))

## Part 1: Scalar Pandas UDF

In [ ]:
from pyspark.sql.functions import pandas_udf

# Test data
df = spark.range(100_000) \
    .withColumn("amount",   (F.col("id") % 500 + 1).cast("double")) \
    .withColumn("category", F.element_at(
        F.array(F.lit("electronics"), F.lit("clothing"), F.lit("food")),
        (F.col("id") % 3 + 1).cast("int")))

# Scalar Pandas UDF: Series in, Series out
# Decorated with @pandas_udf and the return type
@pandas_udf(DoubleType())
def compute_discount(amount: pd.Series) -> pd.Series:
    """Apply a 10% discount for amounts > 200, else 5%."""
    return amount * np.where(amount > 200, 0.10, 0.05)

@pandas_udf(StringType())
def normalize_text(text: pd.Series) -> pd.Series:
    """Clean and normalize text: lower + strip."""
    return text.str.lower().str.strip()

result = df \
    .withColumn("discount",    compute_discount(F.col("amount"))) \
    .withColumn("final_price", F.col("amount") - F.col("discount")) \
    .withColumn("cat_clean",   normalize_text(F.col("category")))

print("Scalar Pandas UDF output:")
result.show(5)

# Performance comparison: regular UDF vs pandas UDF
from pyspark.sql.functions import udf

@udf(DoubleType())
def compute_discount_regular(amount):
    if amount is None:
        return None
    return amount * (0.10 if amount > 200 else 0.05)

t0 = time.time()
df.withColumn("discount", compute_discount_regular(F.col("amount"))).count()
regular_time = time.time() - t0

t0 = time.time()
df.withColumn("discount", compute_discount(F.col("amount"))).count()
pandas_time = time.time() - t0

t0 = time.time()
df.withColumn("discount", F.when(F.col("amount") > 200, F.col("amount") * 0.10)
                           .otherwise(F.col("amount") * 0.05)).count()
builtin_time = time.time() - t0

print(f"\nPerformance (100K rows):")
print(f"  Regular UDF:  {regular_time:.3f}s")
print(f"  Pandas UDF:   {pandas_time:.3f}s  ({regular_time/pandas_time:.1f}x faster)")
print(f"  Built-in:     {builtin_time:.3f}s  ({regular_time/builtin_time:.1f}x faster)")

## Part 2: Scalar Iterator Pandas UDF — Expensive Model Loading

In [ ]:
# Iterator Pandas UDF: load model ONCE per executor, apply to all batches
# Use when model loading is expensive (ML model, tokenizer, DB connection)
from typing import Iterator

@pandas_udf(DoubleType())
def score_with_model(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:
    """
    Load a 'model' once per executor partition, then score all batches.
    In production: replace with actual model.load() call.
    """
    # This block runs ONCE per executor (not once per row or batch)
    print("Loading model... (runs once per executor)")
    # model = joblib.load("/models/price_scorer.pkl")  # in production
    # For demo: use a simple scaling function as our "model"
    def model_predict(series):
        return series * 0.85 + 10  # fake model: score = amount × 0.85 + 10

    for batch in iterator:  # process each Arrow batch
        yield model_predict(batch)

result2 = df.withColumn("score", score_with_model(F.col("amount")))
result2.select("id", "amount", "score").show(5)
print("Iterator UDF: model loaded once per executor partition")

## Part 3: applyInPandas — Group-Level Transforms

In [ ]:
# applyInPandas: each group's rows passed as a full pandas DataFrame
# Perfect for: normalization per group, per-group model inference

output_schema = StructType([
    StructField("id",              LongType()),
    StructField("category",        StringType()),
    StructField("amount",          DoubleType()),
    StructField("amount_zscore",   DoubleType()),   # z-score within category
    StructField("amount_pct_rank", DoubleType()),   # percentile within category
    StructField("is_outlier",      BooleanType()),  # amount > mean + 2σ
])

def normalize_per_category(group_df: pd.DataFrame) -> pd.DataFrame:
    """Per-group normalization — requires all rows of the group simultaneously."""
    mean = group_df["amount"].mean()
    std  = group_df["amount"].std()

    group_df["amount_zscore"]   = (group_df["amount"] - mean) / (std if std > 0 else 1)
    group_df["amount_pct_rank"] = group_df["amount"].rank(pct=True)
    group_df["is_outlier"]      = (group_df["amount"] - mean).abs() > 2 * std

    return group_df[["id", "category", "amount", "amount_zscore", "amount_pct_rank", "is_outlier"]]

# Small sample to demonstrate (applyInPandas moves all group data to one task)
sample = df.limit(3000)

normalized = sample.groupby("category").applyInPandas(
    normalize_per_category,
    schema=output_schema
)

print("Per-category normalization with applyInPandas:")
normalized.orderBy("category", "id").show(10)

print("\nOutlier count per category:")
normalized.groupBy("category").agg(
    F.count("*").alias("total"),
    F.sum(F.col("is_outlier").cast("int")).alias("outliers")
).show()

## Part 4: Arrow — toPandas() and createDataFrame()

In [ ]:
# Arrow also accelerates toPandas() and createDataFrame()
# These cross the JVM↔Python boundary — Arrow makes them much faster

sample_df = df.limit(100_000)

# With Arrow enabled (already set in SparkSession builder)
t0 = time.time()
pdf = sample_df.toPandas()
arrow_time = time.time() - t0
print(f"toPandas() with Arrow: {arrow_time:.3f}s, shape: {pdf.shape}")

# Disable Arrow temporarily for comparison
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
t0 = time.time()
pdf2 = sample_df.toPandas()
no_arrow_time = time.time() - t0
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

print(f"toPandas() without Arrow: {no_arrow_time:.3f}s")
print(f"Arrow speedup: {no_arrow_time/arrow_time:.1f}x")

# createDataFrame from pandas also benefits from Arrow
big_pdf = pdf.copy()
t0 = time.time()
sdf = spark.createDataFrame(big_pdf)
sdf.count()  # trigger
create_arrow = time.time() - t0
print(f"\ncreateDataFrame() with Arrow: {create_arrow:.3f}s")

## Exercises

1. Write a Pandas UDF that computes a 7-day rolling average of `amount` using pandas `.rolling(7).mean()`. Hint: this requires all rows in order — use `applyInPandas` with a date sort inside the group function.
2. Write an Iterator Pandas UDF that "loads" a lookup dictionary (simulating an expensive lookup table) and uses it to map `customer_id → tier` for each row.
3. Compare performance of three approaches for computing `log(amount + 1)` on 1M rows: (a) `F.log(F.col("amount") + 1)`, (b) regular Python UDF, (c) Pandas UDF. Record wall times.
4. When would you use `applyInPandas` vs a Window function? Give a concrete example where `applyInPandas` is unavoidable.
5. Explain the memory risk of `applyInPandas` with a high-cardinality group key (e.g., `user_id` with 10M unique values). What happens if one group has 1B rows?